# Landing diaria - Precipitacion Salto Grande

Descarga los dias faltantes disponibles en la API de Salto Grande y guarda un JSON por dia en Raw.

In [ ]:
from __future__ import annotations

import csv
import json
import time
import warnings
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable
from zoneinfo import ZoneInfo

import requests
from requests.adapters import HTTPAdapter
from urllib3.exceptions import InsecureRequestWarning
from urllib3.util.retry import Retry

warnings.filterwarnings('ignore', category=InsecureRequestWarning)

ENDPOINT = 'https://www.saltogrande.org/ws.php'
NAMESPACE = 'https://www.saltogrande.org/ws.php'
SOAP_ENV = 'http://schemas.xmlsoap.org/soap/envelope/'
RAW_DIR = Path('/Volumes/weather/raw/sg_volume/json/daily')
STATIONS_FILE = Path('/Volumes/weather/raw/sg_volume/sg_estaciones_activas/estaciones_activas.csv')
LOCAL_TZ = ZoneInfo('America/Montevideo')
VARIABLE = 'P'
DEFAULT_TIMEOUT = 60

try:
    dbutils.widgets.text('available_days', '30')
    dbutils.widgets.dropdown('force_reload', 'false', ['false', 'true'])
    dbutils.widgets.text('request_sleep_seconds', '0.2')
    available_days = int(dbutils.widgets.get('available_days'))
    force_reload = dbutils.widgets.get('force_reload').lower() == 'true'
    request_sleep_seconds = float(dbutils.widgets.get('request_sleep_seconds'))
except Exception:
    available_days = 30
    force_reload = False
    request_sleep_seconds = 0.2

print(f'available_days={available_days}, force_reload={force_reload}, request_sleep_seconds={request_sleep_seconds}')

In [ ]:
class SaltoGrandeApiError(RuntimeError):
    pass


def create_session() -> requests.Session:
    retry = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=['POST'],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session = requests.Session()
    session.verify = False
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    return session


def find_first_by_localname(elem: ET.Element, local: str):
    for child in elem.iter():
        tag = child.tag
        if tag == local or tag.endswith('}' + local):
            return child
    return None


def findall_children_by_localname(elem: ET.Element, local: str) -> list[ET.Element]:
    return [
        child
        for child in list(elem)
        if child.tag == local or child.tag.endswith('}' + local)
    ]


def build_hidroserie_payload(id_estacion: str, variable: str, fecha_desde: str, fecha_hasta: str) -> str:
    return f'''<?xml version="1.0" encoding="utf-8"?>
<soapenv:Envelope xmlns:soapenv="{SOAP_ENV}" xmlns:tns="{NAMESPACE}">
  <soapenv:Body>
    <tns:HidroSerieHistorica>
      <idEstacion>{id_estacion}</idEstacion>
      <variable>{variable}</variable>
      <fechaDesde>{fecha_desde}</fechaDesde>
      <fechaHasta>{fecha_hasta}</fechaHasta>
    </tns:HidroSerieHistorica>
  </soapenv:Body>
</soapenv:Envelope>'''


def parse_date(value: Any) -> date | None:
    if value is None:
        return None
    text = str(value).strip()
    if len(text) < 10:
        return None
    try:
        return date.fromisoformat(text[:10])
    except ValueError:
        return None


def parse_float(value: Any) -> float | None:
    if value is None:
        return None
    text = str(value).strip().replace(',', '.')
    if text == '':
        return None
    try:
        return float(text)
    except ValueError:
        return None


def fetch_precipitacion(session: requests.Session, id_estacion: str, fecha_desde: str, fecha_hasta: str) -> list[dict[str, Any]]:
    headers = {
        'Content-Type': 'text/xml; charset=utf-8',
        'SOAPAction': f'{NAMESPACE}#HidroSerieHistorica',
    }
    payload = build_hidroserie_payload(id_estacion, VARIABLE, fecha_desde, fecha_hasta)
    response = session.post(ENDPOINT, data=payload.encode('utf-8'), headers=headers, timeout=DEFAULT_TIMEOUT)
    response.raise_for_status()

    root = ET.fromstring(response.content)
    body = root.find('.//soap:Body', namespaces={'soap': SOAP_ENV})
    if body is None:
        return []

    response_node = find_first_by_localname(body, 'HidroSerieHistoricaResponse')
    if response_node is None:
        return []

    ret = find_first_by_localname(response_node, 'return')
    if ret is None:
        return []

    records = []
    for item in findall_children_by_localname(ret, 'item'):
        fecha = find_first_by_localname(item, 'Fecha')
        valor = find_first_by_localname(item, 'Valor')
        records.append(
            {
                'Fecha': fecha.text if fecha is not None else None,
                'Id_Estacion': id_estacion,
                'P': parse_float(valor.text if valor is not None else None),
            }
        )
    return records

In [ ]:
def load_active_precipitation_stations(path: Path = STATIONS_FILE) -> list[dict[str, Any]]:
    if not path.exists():
        raise SaltoGrandeApiError(f'No existe el archivo de estaciones: {path}')

    stations = []
    seen = set()
    with path.open(newline='', encoding='utf-8-sig') as csv_file:
        for row in csv.DictReader(csv_file):
            station_id = (row.get('Id') or '').strip()
            variables = (row.get('Variables') or '').split(',')
            if not station_id or VARIABLE not in {value.strip() for value in variables}:
                continue
            if station_id in seen:
                continue
            seen.add(station_id)
            stations.append(
                {
                    'Id_Estacion': station_id,
                    'Nombre': row.get('Nombre'),
                    'Latitud': parse_float(row.get('Latitud')),
                    'Longitud': parse_float(row.get('Longitud')),
                }
            )

    if not stations:
        raise SaltoGrandeApiError('No se encontraron estaciones activas con variable P')
    return stations


def expected_dates() -> list[date]:
    end_date = datetime.now(LOCAL_TZ).date() - timedelta(days=1)
    start_date = end_date - timedelta(days=max(available_days, 1) - 1)
    return [start_date + timedelta(days=offset) for offset in range((end_date - start_date).days + 1)]


def raw_filename(target_date: date) -> str:
    return f"SG_P_{target_date.strftime('%Y_%m_%d')}.json"


def existing_raw_dates() -> set[date]:
    if not RAW_DIR.exists():
        return set()
    dates = set()
    for path in RAW_DIR.glob('SG_P_*.json'):
        try:
            dates.add(datetime.strptime(path.stem.replace('SG_P_', ''), '%Y_%m_%d').date())
        except ValueError:
            continue
    return dates


def missing_dates() -> list[date]:
    dates = expected_dates()
    if force_reload:
        return dates
    existing = existing_raw_dates()
    return [target_date for target_date in dates if target_date not in existing]


# La API SOAP de Salto Grande rechaza (HTTP 500, "Diferencia de fechas mayor a 31
# dias") cualquier fechaDesde/fechaHasta que abarque mas de 31 dias. Si no se
# chequea, el notebook escribe archivos vacios para todos los dias faltantes
# antes de fallar, y el reintento automatico los toma como "ya completos" -
# enmascarando el fallo. Por eso partimos el rango en bloques de <=31 dias.
MAX_RANGE_DAYS = 31


def chunk_date_ranges(target_dates: list[date]) -> list[list[date]]:
    ordenadas = sorted(target_dates)
    bloques: list[list[date]] = []
    bloque_actual: list[date] = []
    for fecha in ordenadas:
        if bloque_actual and (fecha - bloque_actual[0]).days + 1 > MAX_RANGE_DAYS:
            bloques.append(bloque_actual)
            bloque_actual = []
        bloque_actual.append(fecha)
    if bloque_actual:
        bloques.append(bloque_actual)
    return bloques


def write_daily_json(target_date: date, records: list[dict[str, Any]]) -> None:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    path = RAW_DIR / raw_filename(target_date)
    path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'OK {path.name}: registros={len(records)}')

In [ ]:
def run() -> None:
    target_dates = missing_dates()
    if not target_dates:
        print('No hay dias faltantes para descargar en la ventana disponible')
        return

    stations = load_active_precipitation_stations()
    station_by_id = {station['Id_Estacion']: station for station in stations}
    target_date_set = set(target_dates)
    extracted_at = datetime.now(timezone.utc).isoformat()

    bloques = chunk_date_ranges(target_dates)
    print(f'Descargando SG P; estaciones={len(stations)}; dias={len(target_dates)}; bloques={len(bloques)}')

    session = create_session()
    grouped: dict[date, list[dict[str, Any]]] = defaultdict(list)
    errors = []

    for bloque_idx, bloque in enumerate(bloques, start=1):
        fecha_desde = min(bloque).isoformat()
        fecha_hasta = max(bloque).isoformat()
        print(f'Bloque {bloque_idx}/{len(bloques)}: {fecha_desde} a {fecha_hasta} ({len(bloque)} dias)')

        for index, station in enumerate(stations, start=1):
            station_id = station['Id_Estacion']
            try:
                records = fetch_precipitacion(session, station_id, fecha_desde, fecha_hasta)
            except Exception as exc:
                print(f'Error estacion={station_id}: {exc}')
                errors.append(station_id)
                continue

            for record in records:
                record_date = parse_date(record.get('Fecha'))
                if record_date not in target_date_set:
                    continue
                metadata = station_by_id.get(station_id, {})
                grouped[record_date].append(
                    {
                        **record,
                        'Nombre': metadata.get('Nombre'),
                        'Latitud': metadata.get('Latitud'),
                        'Longitud': metadata.get('Longitud'),
                        'source_api': 'salto_grande_hidroserie_historica',
                        'extracted_at': extracted_at,
                    }
                )

            if index % 10 == 0 or index == len(stations):
                print(f'Estaciones consultadas: {index}/{len(stations)}')
            if request_sleep_seconds > 0:
                time.sleep(request_sleep_seconds)

    for target_date in target_dates:
        daily_records = sorted(grouped.get(target_date, []), key=lambda item: str(item.get('Id_Estacion')))
        write_daily_json(target_date, daily_records)

    if errors:
        raise SaltoGrandeApiError(f'Estaciones con error: {len(errors)} de {len(stations)}')


try:
    run()
except SaltoGrandeApiError as exc:
    raise SystemExit(f'ERROR SG: {exc}')